In [1]:
from collections import Counter

import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.special import gammaln
from scipy.stats import poisson

In [2]:
MIN_OPPONENTS_TO_CONSIDER = 1
MIN_GAMES_TO_CONSIDER = 1
MIN_DATE = "2023-01-01"

NUM_SIMULATIONS = 100_000

In [3]:
columns = [
    "date",
    "home_team",
    "away_team",
    "home_score",
    "away_score",
    "tournament",
    "neutral",
]

data = pd.read_csv("../data/results.csv")
data = data[data["date"] >= MIN_DATE][columns]

# temporal weight
data["date"] = pd.to_datetime(data["date"])
data["weeks_since_min_date"] = (data["date"] - pd.to_datetime(MIN_DATE)).dt.days // 7
max_date = data["date"].max()
weeks_size = (max_date - pd.to_datetime(MIN_DATE)).days // 7
data["game_weight"] = 0.3 + 0.7 * (data["weeks_since_min_date"] / weeks_size)

# BFS approach to find all teams that have played at least MIN_GAMES_TO_CONSIDER games
teams_to_evaluate = {"Brazil"}
teams_to_consider = set()
while len(teams_to_evaluate) > 0:
    team = teams_to_evaluate.pop()
    teams_to_consider.add(team)

    matches = data[(data["home_team"] == team) | (data["away_team"] == team)]

    unique_opponents = set(matches["home_team"].unique())
    unique_opponents.update(matches["away_team"].unique())

    unique_opponents.discard(team)
    if len(unique_opponents) <= MIN_OPPONENTS_TO_CONSIDER:
        teams_to_consider.discard(team)
        continue

    teams_to_evaluate.update(unique_opponents)

    teams_to_evaluate = teams_to_evaluate - teams_to_consider

data = data[
    (data["home_team"].isin(teams_to_consider))
    & (data["away_team"].isin(teams_to_consider))
]

data

,date,home_team,away_team,home_score,away_score,tournament,neutral,weeks_since_min_date,game_weight
45770,2023-01-02,Philippines,Indonesia,1,2,AFF Championship,False,0,0.300000
45771,2023-01-02,Thailand,Cambodia,3,1,AFF Championship,False,0,0.300000
45772,2023-01-03,Vietnam,Myanmar,3,0,AFF Championship,False,0,0.300000
45773,2023-01-03,Malaysia,Singapore,4,1,AFF Championship,False,0,0.300000
45774,2023-01-06,Iraq,Oman,0,0,Gulf Cup,False,0,0.300000
...,...,...,...,...,...,...,...,...,...
49066,2026-01-18,Grenada,Jamaica,0,1,Friendly,False,159,0.995625
49067,2026-01-18,Morocco,Senegal,0,1,African Cup of Nations,False,159,0.995625
49068,2026-01-22,Panama,Mexico,0,1,Friendly,False,159,0.995625
49069,2026-01-25,Bolivia,Mexico,0,1,Friendly,False,160,1.000000


In [4]:
# filter out teams that have played less than MIN_GAMES_TO_CONSIDER games
home_games = (
    data.groupby("home_team")
    .agg(
        home_games=("home_score", "size"),
        home_goals=("home_score", "sum"),
    )
    .sort_values(by="home_games", ascending=False)
    .reset_index()
)

away_games = (
    data.groupby("away_team")
    .agg(
        away_games=("away_score", "size"),
        away_goals=("away_score", "sum"),
    )
    .sort_values(by="away_games", ascending=False)
    .reset_index()
)

games = away_games.merge(
    home_games, left_on="away_team", right_on="home_team", how="outer"
)

games["home_games"] = games["home_games"].fillna(0).astype(int)
games["away_games"] = games["away_games"].fillna(0).astype(int)
games["total_games"] = games["home_games"] + games["away_games"]

games["home_goals"] = games["home_goals"].fillna(0).astype(int)
games["away_goals"] = games["away_goals"].fillna(0).astype(int)
games["total_goals"] = games["home_goals"] + games["away_goals"]

games = games.sort_values(by="total_games", ascending=False, ignore_index=True)

games["team"] = np.where(
    games["home_team"].isna(), games["away_team"], games["home_team"]
)

games = games[["team", "total_games", "total_goals"]]
games = games[games["total_games"] >= MIN_GAMES_TO_CONSIDER]
teams = games["team"].tolist()

data = data[(data["home_team"].isin(teams)) & (data["away_team"].isin(teams))]

data["home_idx"] = data["home_team"].apply(lambda x: teams.index(x))
data["away_idx"] = data["away_team"].apply(lambda x: teams.index(x))

print(f"Number of teams: {len(teams)}")

Number of teams: 239


In [5]:
# Extract constant values before function call
_home_idx = data["home_idx"].values
_away_idx = data["away_idx"].values
_home_score = data["home_score"].values
_away_score = data["away_score"].values
_has_home_effect = ~data["neutral"].values
_game_weight = data["game_weight"].values

# constant terms
_n_teams = len(teams)
_home_log_factorial = gammaln(_home_score + 1)
_away_log_factorial = gammaln(_away_score + 1)

# masks
_mask_00 = (_home_score == 0) & (_away_score == 0)
_mask_10 = (_home_score == 1) & (_away_score == 0)
_mask_01 = (_home_score == 0) & (_away_score == 1)
_mask_11 = (_home_score == 1) & (_away_score == 1)

In [6]:
bounds = [(1e-6, None)] * (_n_teams - 1)  # strengths (teams 1..N-1)
bounds.append((1e-6, None))  # home_effect
bounds.append((-1.0 + 1e-6, 1.0 - 1e-6))  # rho

x0 = np.ones(len(teams) + 1)
x0[-1] = 0

In [7]:
def log_likelihood_with_grad(params):
    home_effect = params[-2]
    rho = params[-1]
    strengths = np.empty(_n_teams)
    strengths[0] = 1.0
    strengths[1:] = params[:-2]

    home_force = strengths[_home_idx] + home_effect * _has_home_effect
    away_force = strengths[_away_idx]

    home_lambda = home_force / away_force
    away_lambda = away_force / home_force
    log_ratio = np.log(home_lambda)

    ll_home = _home_score * log_ratio - home_lambda - _home_log_factorial
    ll_away = -_away_score * log_ratio - away_lambda - _away_log_factorial

    tau = np.ones(len(_home_score))
    tau[_mask_00] = 1 - rho
    tau[_mask_10] = 1 + away_lambda[_mask_10] * rho
    tau[_mask_01] = 1 + home_lambda[_mask_01] * rho
    tau[_mask_11] = 1 - rho

    if np.any(tau <= 0):
        return 1e10, np.zeros(len(params))

    nll = -float((_game_weight * (ll_home + ll_away + np.log(tau))).sum())

    # --- Derivadas de log(tau) em relação a f_h e f_a ---
    dlogtau_dfh = np.zeros(len(_home_score))
    dlogtau_dfh[_mask_10] = (
        -away_lambda[_mask_10] * rho / (home_force[_mask_10] * tau[_mask_10])
    )
    dlogtau_dfh[_mask_01] = rho / (away_force[_mask_01] * tau[_mask_01])

    dlogtau_dfa = np.zeros(len(_home_score))
    dlogtau_dfa[_mask_10] = rho / (home_force[_mask_10] * tau[_mask_10])
    dlogtau_dfa[_mask_01] = (
        -home_lambda[_mask_01] * rho / (away_force[_mask_01] * tau[_mask_01])
    )

    # --- Gradiente por jogo em relação a f_h e f_a ---
    score_diff = _home_score - _away_score
    dnll_dfh = -_game_weight * (
        score_diff / home_force
        - 1 / away_force
        + away_force / home_force**2
        + dlogtau_dfh
    )
    dnll_dfa = -_game_weight * (
        -score_diff / away_force
        + home_force / away_force**2
        - 1 / home_force
        + dlogtau_dfa
    )

    # --- Acumular nos parâmetros de strength ---
    strength_grad = np.zeros(_n_teams)
    np.add.at(strength_grad, _home_idx, dnll_dfh)
    np.add.at(strength_grad, _away_idx, dnll_dfa)

    grad = np.empty(len(params))
    grad[:-2] = strength_grad[1:]  # pula team 0 (fixo em 1.0)

    # home_effect: ∂f_h/∂he = is_home
    grad[-2] = (dnll_dfh * _has_home_effect).sum()

    # rho
    dlogtau_drho = np.zeros(len(_home_score))
    dlogtau_drho[_mask_00] = -1 / tau[_mask_00]
    dlogtau_drho[_mask_10] = away_lambda[_mask_10] / tau[_mask_10]
    dlogtau_drho[_mask_01] = home_lambda[_mask_01] / tau[_mask_01]
    dlogtau_drho[_mask_11] = -1 / tau[_mask_11]
    grad[-1] = -(_game_weight * dlogtau_drho).sum()

    return nll, grad

In [8]:
res = minimize(
    fun=log_likelihood_with_grad,
    x0=x0,
    method="L-BFGS-B",
    jac=True,
    bounds=bounds,
    options={"maxiter": 10_000, "maxfun": 500_000},
)

res.success, res.message, res.fun

(True,
 'CONVERGENCE: RELATIVE REDUCTION OF F <= FACTR*EPSMCH',
 5997.618277822008)

In [9]:
home_effect = res.x[-2]
rho = res.x[-1]
strengths = np.concatenate([[1.0], res.x[:-2]])

games["idx"] = games["team"].apply(lambda x: teams.index(x))
games["strength"] = strengths[games["idx"]]
games.drop(columns=["idx"], inplace=True)
games.sort_values(by="strength", ascending=False, ignore_index=True, inplace=True)

games.head(20)

,team,total_games,total_goals,strength
0,Spain,37,101,3.408815
1,Argentina,35,73,2.892977
2,Colombia,38,71,2.783271
3,Norway,30,82,2.496233
4,Portugal,36,96,2.459665
5,France,36,80,2.437896
6,Germany,36,74,2.420364
7,Netherlands,36,89,2.417105
8,England,37,81,2.387333
9,Brazil,33,54,2.348341


In [10]:
# played at neutral ground
home = "Brazil"
away = "Croatia"

home_strength = games.loc[games["team"] == home, "strength"].values[0]
away_strength = games.loc[games["team"] == away, "strength"].values[0]
rho = res.x[-1]

home_lambda = home_strength / away_strength
away_lambda = away_strength / home_strength

max_goals = 10
h_probs = poisson.pmf(np.arange(max_goals + 1), home_lambda)
a_probs = poisson.pmf(np.arange(max_goals + 1), away_lambda)
prob_matrix = np.outer(h_probs, a_probs)

# Dixon-Coles correction
prob_matrix[0, 0] *= 1 - home_lambda * away_lambda * rho
prob_matrix[1, 0] *= 1 + away_lambda * rho
prob_matrix[0, 1] *= 1 + home_lambda * rho
prob_matrix[1, 1] *= 1 - rho
prob_matrix /= prob_matrix.sum()

# sample from the adjusted distribution
flat = prob_matrix.flatten()
samples = np.random.choice(len(flat), size=NUM_SIMULATIONS, p=flat)
home_goals = samples // (max_goals + 1)
away_goals = samples % (max_goals + 1)

home_wins = (home_goals > away_goals).sum() / NUM_SIMULATIONS
away_wins = (home_goals < away_goals).sum() / NUM_SIMULATIONS
draws = (home_goals == away_goals).sum() / NUM_SIMULATIONS

most_common_score = Counter(zip(home_goals, away_goals, strict=False)).most_common(1)[
    0
][0]

home_goals, away_goals = most_common_score

print(f"{home} strength: {home_strength:.2f}")
print(f"{away} strength: {away_strength:.2f}")

print()
print(f"{home} wins: {home_wins:.2%}")
print(f"Draws: {draws:.2%}")
print(f"{away} wins: {away_wins:.2%}")

print()
print(f"Most common score: {home} {home_goals} - {away_goals} {away}")

Brazil strength: 2.35
Croatia strength: 1.87

Brazil wins: 45.79%
Draws: 31.70%
Croatia wins: 22.52%

Most common score: Brazil 1 - 0 Croatia
